### Full Name of member — Yoonjae Lee

**Section A — Group 6**  
Source image: `Images/Group_6.png`  
Name: **Yoonjae** / **Lee** (first letter of each part capitalised in the labels below).

Extraction uses **array slicing** on the image (`image[y1:y2, x1:x2]`) after building row/column boundaries for the 6×9 letter grid. A small **margin slice** removes the brown grid lines at cell edges; **equal letter size** is enforced by **center cropping** each patch to the same height and width using further slices only.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# --- paths ---
ROOT = Path(".")
IMAGE_PATH = ROOT / "Images" / "Group_6.png"

img = np.array(Image.open(IMAGE_PATH).convert("RGB"))
H, W = img.shape[0], img.shape[1]
ROWS, COLS = 6, 9


def split_lengths(total: int, n: int) -> list[int]:
    """Split `total` into `n` integer lengths differing by at most 1 (remainder spread from the top)."""
    base, rem = total // n, total % n
    return [base + (1 if i < rem else 0) for i in range(n)]


row_heights = split_lengths(H, ROWS)
col_widths = split_lengths(W, COLS)
y_edges = [0] + list(np.cumsum(row_heights))
x_edges = [0] + list(np.cumsum(col_widths))


def cell(r: int, c: int) -> np.ndarray:
    """Slice one grid cell (row r, column c) with Lab-style [y1:y2, x1:x2] indexing."""
    return img[y_edges[r] : y_edges[r + 1], x_edges[c] : x_edges[c + 1]]


def strip_border(patch: np.ndarray, margin: int) -> np.ndarray:
    """Remove a band of pixels on each side (grid lines) using slicing only."""
    if margin <= 0:
        return patch
    h, w = patch.shape[0], patch.shape[1]
    if h <= 2 * margin or w <= 2 * margin:
        return patch
    return patch[margin:-margin, margin:-margin]


def center_crop(patch: np.ndarray, out_h: int, out_w: int) -> np.ndarray:
    """Crop to a fixed size from the centre using slicing only."""
    h, w = patch.shape[0], patch.shape[1]
    y0 = max(0, (h - out_h) // 2)
    x0 = max(0, (w - out_w) // 2)
    return patch[y0 : y0 + out_h, x0 : x0 + out_w]


def letter(r: int, c: int, margin: int, out_h: int, out_w: int) -> np.ndarray:
    p = strip_border(cell(r, c), margin)
    return center_crop(p, out_h, out_w)


# Grid coordinates (row, col) for "Yoonjae" and "Lee" — verified on Group_6.png
# Y(0,7) o(1,1)×2 n(0,2) j(3,8) a(1,2) e(5,2) | L(5,1) e(5,2)×2
FIRST_COORDS = [(0, 7), (1, 1), (1, 1), (0, 2), (3, 8), (1, 2), (5, 2)]
LAST_COORDS = [(5, 1), (5, 2), (5, 2)]

MARGIN = 6  # trims brown grid lines; adjust 4–8 if needed
test_patches = [strip_border(cell(r, c), MARGIN) for r, c in set(FIRST_COORDS + LAST_COORDS)]
OUT_H = min(p.shape[0] for p in test_patches)
OUT_W = min(p.shape[1] for p in test_patches)

first_letters = [letter(r, c, MARGIN, OUT_H, OUT_W) for r, c in FIRST_COORDS]
last_letters = [letter(r, c, MARGIN, OUT_H, OUT_W) for r, c in LAST_COORDS]


def show_name_row(title: str, name_label: str, patches: list[np.ndarray]) -> None:
    n = len(patches)
    fig, axes = plt.subplots(1, n, figsize=(2 * n, 2.8))
    if n == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=12, y=1.02)
    for ax, patch in zip(axes, patches):
        ax.imshow(np.clip(patch, 0, 255).astype(np.uint8))
        ax.axis("off")
    fig.text(0.5, 0.02, name_label, ha="center", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


print(
    "My name is Yoonjae Lee, and I am slicing the image Group_6.png to extract my name."
)
print(f"Equal letter size (H×W): {OUT_H}×{OUT_W}")

show_name_row("i. First name", "First name: Yoonjae", first_letters)
show_name_row("ii. Last name", "Last name: Lee", last_letters)